# Image captioning with CNN and LSTM

A caption generator combines the two kinds of networks seen so far: a CNN turns the image into a feature vector, and an LSTM uses this vector as its initial state to generate a sentence, one word at a time. This notebook trains the LSTM on the Flickr8k dataset, using a pre-trained ResNet-50 as a fixed feature extractor.

Use a GPU runtime: the dataset has about 1 GB, and it is stored in the disk of the Colab VM, not in your Drive.

**Aprendizado Profundo para Reconhecimento Visual** — Ifes Campus Serra
Autor: Thiago M. Paixão

In [ ]:
# If True, test is running on Colab. Otherwise, test is assumed to be offline.
TEST_ON_COLAB = True
FOLDERNAME = 'Colab Notebooks/Aprendizado Profundo para Reconhecimento Visual/07_rnn'  # only used if TEST_ON_COLAB is True

assert not (FOLDERNAME is None and TEST_ON_COLAB), "FOLDERNAME has to be set if TEST_ON_COLAB is True"

In [ ]:
# Create dataset directory
import os

if TEST_ON_COLAB:
    # This mounts your Google Drive to the Colab VM.
    from google.colab import drive

    drive.mount('/content/drive')
    os.chdir(f'/content/drive/My Drive/{FOLDERNAME}')

## Dataset

Flickr8k has about 8000 images, each with five captions written by people. The dataset is intended for research use. The download below comes from a public copy, not from the original authors.

In [ ]:
import random
import re
from collections import Counter

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torchvision import models
from torchvision.datasets.utils import download_and_extract_archive

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)

data_dir = "/tmp/flickr8k"
image_dir = f"{data_dir}/Flicker8k_Dataset"  # this is the folder name in the archive
base_url = "https://github.com/jbrownlee/Datasets/releases/download/Flickr8k"

if not os.path.isdir(image_dir):
    download_and_extract_archive(f"{base_url}/Flickr8k_Dataset.zip", data_dir, remove_finished=True)
    download_and_extract_archive(f"{base_url}/Flickr8k_text.zip", data_dir, remove_finished=True)


def read_split(name):
    with open(f"{data_dir}/Flickr_8k.{name}Images.txt") as f:
        return [line.strip() for line in f if line.strip()]


splits = {name: read_split(name) for name in ("train", "dev", "test")}

captions = {}  # image file name -> list of captions, each one a list of words
with open(f"{data_dir}/Flickr8k.token.txt") as f:
    for line in f:
        image_id, text = line.strip().split("\t")
        image_name = image_id.split("#")[0]
        captions.setdefault(image_name, []).append(re.findall(r"[a-z]+", text.lower()))

print({name: len(images) for name, images in splits.items()})
print(captions[splits["train"][0]][0])

## Vocabulary

Captions have different lengths, so here we do need the special tokens that the toy tasks could avoid:

- `<pad>` fills the short captions so that a batch forms a single tensor. The loss ignores it.
- `<sos>` is the first decoder input.
- `<eos>` is the last target. By predicting it, the model decides when the sentence ends.
- `<unk>` replaces rare words, which appear too few times for the model to learn them.

In [ ]:
pad, sos, eos, unk = 0, 1, 2, 3
max_length = 25  # longer captions are cut

counts = Counter(word for name in splits["train"] for caption in captions[name] for word in caption)
vocabulary = ["<pad>", "<sos>", "<eos>", "<unk>"] + sorted(word for word, count in counts.items() if count >= 5)
word_to_index = {word: i for i, word in enumerate(vocabulary)}
print(f"Vocabulary size: {len(vocabulary)}")


def encode(caption):
    tokens = [word_to_index.get(word, unk) for word in caption[:max_length]]
    tokens = [sos] + tokens + [eos]
    return tokens + [pad] * (max_length + 2 - len(tokens))


def decode(tokens):
    words = []
    for token in tokens:
        if token == eos:
            break
        words.append(vocabulary[token])
    return " ".join(words)


print(encode(captions[splits["train"][0]][0]))

## Image features

The CNN is pre-trained on ImageNet and stays frozen. Removing its classification layer leaves a vector of 2048 features for each image. Since these vectors never change, we compute them once, and training only touches the LSTM.

In [ ]:
weights = models.ResNet50_Weights.IMAGENET1K_V2
cnn = models.resnet50(weights=weights)
cnn.fc = nn.Identity()  # keeps the 2048 features that would go into the classifier
cnn = cnn.to(device).eval()
preprocess = weights.transforms()


class ImageFolderList(torch.utils.data.Dataset):
    def __init__(self, names):
        self.names = names

    def __len__(self):
        return len(self.names)

    def __getitem__(self, i):
        return preprocess(Image.open(f"{image_dir}/{self.names[i]}").convert("RGB"))


@torch.no_grad()
def extract_features(names):
    loader = torch.utils.data.DataLoader(ImageFolderList(names), batch_size=64, num_workers=2)
    return torch.cat([cnn(images.to(device)) for images in loader])


features = {name: extract_features(images) for name, images in splits.items()}  # (num_images, 2048)
print({name: tuple(value.shape) for name, value in features.items()})


def make_pairs(name):
    """Returns the image index and the encoded caption of each of the five captions of each image."""
    indices, tokens = [], []
    for i, image_name in enumerate(splits[name]):
        for caption in captions[image_name]:
            indices.append(i)
            tokens.append(encode(caption))
    return torch.tensor(indices, device=device), torch.tensor(tokens, device=device)


train_pairs, dev_pairs = make_pairs("train"), make_pairs("dev")
print(train_pairs[1].shape)  # (num_captions, max_length + 2)

## Model

The image enters the LSTM only once, through the initial states $h_0$ and $c_0$, which are linear projections of the features. As in the Seq2Seq model, the decoder is trained with teacher forcing: it receives the caption shifted by one position and predicts the next word.

In [ ]:
class Captioner(nn.Module):
    def __init__(self, vocabulary_size, feature_size=2048, embedding_size=256, hidden_size=512):
        super().__init__()
        self.init_h = nn.Linear(feature_size, hidden_size)
        self.init_c = nn.Linear(feature_size, hidden_size)
        self.embedding = nn.Embedding(vocabulary_size, embedding_size, padding_idx=pad)
        self.lstm = nn.LSTM(embedding_size, hidden_size, batch_first=True)
        self.dropout = nn.Dropout(0.5)
        self.output = nn.Linear(hidden_size, vocabulary_size)

    def initial_state(self, features):
        h = torch.tanh(self.init_h(self.dropout(features)))
        c = torch.tanh(self.init_c(self.dropout(features)))
        return h.unsqueeze(0), c.unsqueeze(0)  # (N, H) -> (num_layers, N, H), the format of nn.LSTM

    def forward(self, features, tokens):
        """Teacher forcing: returns the logits, of shape (N, T, V)."""
        states, _ = self.lstm(self.embedding(tokens), self.initial_state(features))
        return self.output(self.dropout(states))

    @torch.no_grad()
    def generate(self, features, max_steps=max_length):
        """Greedy generation: returns the predicted tokens, of shape (N, max_steps)."""
        state = self.initial_state(features)
        token = torch.full((features.size(0), 1), sos, device=features.device)
        predictions = []
        for _ in range(max_steps):
            step, state = self.lstm(self.embedding(token), state)
            token = self.output(step).argmax(dim=2)
            predictions.append(token)
        return torch.cat(predictions, dim=1)

## Training

The target is the input shifted by one position. With `ignore_index`, the padding positions do not count in the loss.

In [ ]:
def caption_loss(model, image_features, tokens):
    logits = model(image_features, tokens[:, :-1])  # input: <sos>, w_1, ..., w_n
    targets = tokens[:, 1:]  # target: w_1, ..., w_n, <eos>
    return F.cross_entropy(logits.reshape(-1, logits.size(2)), targets.reshape(-1), ignore_index=pad)


@torch.no_grad()
def evaluate(model, name, pairs):
    model.eval()
    indices, tokens = pairs
    losses = [caption_loss(model, features[name][indices[i:i + 512]], tokens[i:i + 512]).item()
              for i in range(0, len(tokens), 512)]
    model.train()
    return sum(losses) / len(losses)


model = Captioner(len(vocabulary)).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
train_history, dev_history = [], []
indices, tokens = train_pairs
batch_size = 128
for epoch in range(10):
    order = torch.randperm(len(tokens), device=device)
    total = 0
    for i in range(0, len(tokens), batch_size):
        batch = order[i:i + batch_size]
        loss = caption_loss(model, features["train"][indices[batch]], tokens[batch])

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total += loss.item() * len(batch)
    train_history.append(total / len(tokens))
    dev_history.append(evaluate(model, "dev", dev_pairs))
    print(f"Epoch {epoch + 1}, Train loss: {train_history[-1]:.4f}, Dev loss: {dev_history[-1]:.4f}")

plt.plot(train_history, label="Train")
plt.plot(dev_history, label="Dev")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.title("Loss History")
plt.show()

## Generated captions

The images below were never seen during training. At each step the model takes the most likely word and feeds it back, until it produces `<eos>`.

Expect a mix of good and bad captions. With only 6000 training images and a frozen CNN, the model tends to repeat common patterns of the dataset, such as dogs running on grass, and it can describe the scene while getting details wrong.

In [ ]:
model.eval()
random.seed(0)
samples = random.sample(range(len(splits["test"])), 6)
generated = model.generate(features["test"][samples]).tolist()

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for axis, i, tokens_i in zip(axes.flat, samples, generated):
    image_name = splits["test"][i]
    axis.imshow(Image.open(f"{image_dir}/{image_name}"))
    axis.set_title(f"Generated: {decode(tokens_i)}\nHuman: {' '.join(captions[image_name][0])}", fontsize=8, wrap=True)
    axis.axis("off")
plt.tight_layout()
plt.show()